# 01 — Prep, engine gates, part split → **H7**

First of four notebooks (spec §9, `spec/05_corridors_v2_addendum_run_and_alternatives.md` —
read it before touching anything here). This notebook runs everything that happens BEFORE a run
dir exists: the warp (**G2**), the graph-layer selftest (**G4** in miniature), engine equivalence
on v1's own resistance (**G1**), and **step 0a** — the D16 part split with its evidence table.

**It ends at a HARD STOP (H7):** review `audit/audit_objects/multipart_review.csv`, edit the
`proposed` column where the decision rules got it wrong, and fill in the `# reviewed_by:` line.
Notebook 02 refuses to create a run until the file is signed.


In [1]:
# ---- Setup: find the project root, import the shared engines ----------------
# This notebook lives in analyses/northern_connectivity/, below the repo root where config.py
# and the corridor engine modules (corridors_prep / corridor_graph / corridors_core /
# corridors_ensemble) sit. Same bootstrap pattern as analyses/y2y/.
import sys, pathlib
_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
          if (p / "config.py").exists()]
assert _cands, f"config.py not found above {pathlib.Path.cwd()} -- run this notebook from inside the repo"
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))

import importlib
import config
import corridors_prep as cp
import corridor_graph as cg
import corridors_core as cc
import corridors_ensemble as ce
for _m in (config, cp, cg, cc, ce):
    importlib.reload(_m)

KEY = "north"


## 2 · Prep — warp the cost surface to the 300 m routing grid · **gate G2**

One-off (skipped in effect if `movement_cost.tif` is already on disk — `gdalwarp -overwrite`
re-warps identically). `cp.check` asserts CRS/shape/extent fidelity and that `-r near` preserved
the four ordinal classes exactly.


In [2]:
g = cp.grid(KEY)
cp.warp(g)
cp.check(g)


300 m routing grid (ESRI:102008): [-2205300, 1854900, -919800, 3585000]
  4,285 x 5,767 = 24.7 M cells @ 300 m   (lat >= 54.0)
source: Movement_Cost_Layer.tif  EPSG:3347  300 m  24,475 x 24,800  nodata=nan
warping (reads only the source window covering -te) ...
  wrote input_data/corridors_300m/movement_cost.tif (1 MB)
G2 warp fidelity OK: 4,285 x 5,767 @ 300 m, ESRI:102008
  in-corridor cells 9,754,555 of 24,711,595 (39.5% of the window rectangle; the rest is outside the buffered Y2Y cutline)
  class distribution (share of in-corridor cells):
    cost     1:  88.2%  (8,600,713 cells)
    cost    10:   5.4%  (531,002 cells)
    cost   100:   0.3%  (25,325 cells)
    cost  1000:   6.1%  (597,515 cells)


{'valid_cells': 9754555,
 'classes': {1: 8600713, 10: 531002, 100: 25325, 1000: 597515}}

## 3 · Graph-layer self-test · **gate G4 (in miniature) + adjacency handling**


In [3]:
cg.selftest()


corridor_graph.selftest OK — MST size, backbone containment, beta ceiling, adjacency handling, quotient centrality, edge table


True

## 4 · Engine equivalence on the OLD resistance · **gate G1**

The refactor gate — the one that matters. Feeds the engine v1's own frozen resistance on v1's
1 km grid (MST-only, relative band) and requires v1's corridor back at Jaccard ≥ 0.999. Every
other v2 change moves the answer on purpose; this isolates the refactor from the semantics, and
it is what catches the `mcp.traceback` trap.


In [4]:
g1 = cc.gate_g1(KEY)


  merged node: IPCA · Peel Watershed - SMA/WA absorbs Teetł’it Gwinjik (Peel River) (4,143 km² nested)
  merged node: PA · Nj ‘Iinlii” Jjik (Fishing Branch) Habitat Protection Area absorbs Fishing Branch Wilderness Preserve (5,353 km² nested)
  merged node: PA · Neah Conservancy absorbs Ne'ah – Horseranch Range Deadwood Lake Protected Area (2,293 km² nested)
G1: replaying the v1 network on v1's own resistance (1286x1730 @ 1 km, 42 nodes)
  edges      new 41 (12 zero-cost)   v1 41 (12 zero-cost)
  corridor   new 18,188 km²   v1 18,188 km²
  centreline new 1172 cells
  JACCARD vs v1 corridors.tif = 1.0000
  G1 OK — the refactor is behaviour-preserving on identical inputs


## 5 · Step 0a — part split + multipart review proposal (**D16**)

Rasterizes every name at 300 m, labels 8-connected components, applies `part_min_km2`, and
writes `node_parts.csv` / `node_parts.gpkg` plus the PROPOSED `multipart_review.csv` into the
**git-tracked** `analyses/northern_connectivity/audit/audit_objects/`. For each multipart name
the evidence columns (designation, part areas, `min_gap_cells`, `cwd_between_parts`,
`intervening_nodes`, `crosses_cost_1000`) sit beside a treatment proposed by decision rules 1–4.

Note: the PA layer has **no designation attribute**, so existing-PA designations are derived
from `PA_Name` (marked "name-derived") — check those rows especially. Refuses to overwrite an
already-signed review unless `force=True`.


In [5]:
review = cc.node_parts(KEY)
review


Northern BC + Yukon: routing grid 2809x5767 @ 300 m = 9,696,945 routable cells (872,725 km²)
  cropped from the 4285x5767 warped window (66% of its cells) = anchors + 100 km routing buffer
  merged node: IPCA · Peel Watershed - SMA/WA absorbs Teetł’it Gwinjik (Peel River) (4,147 km² nested)
  merged node: PA · Nj ‘Iinlii” Jjik (Fishing Branch) Habitat Protection Area absorbs Fishing Branch Wilderness Preserve (5,355 km² nested)
  merged node: PA · Neah Conservancy absorbs Ne'ah – Horseranch Range Deadwood Lake Protected Area (2,312 km² nested)
nodes: 42 (10 IPCAs + 32 existing PAs >= 200 km²)  [min node size 25 km² = 278 cells @ 300 m]
  dropped 2 IPCA(s) below 25 km² in region: Wëdzey Nähuzhi (Matson Uplands), Łuk Tthe K’ät (Scottie Creek Wetlands)
  node land: 242,540 km² (excluded from the corridor)
node_parts: 42 names -> 58 components, 49 seed parts | 5 multipart names to review
  wrote node_parts.csv, node_parts.gpkg -> analyses/northern_connectivity/audit/audit_objects
  Dene Kʼ

,name_label,kind,designation,n_parts,part_areas_km2,min_gap_cells,max_euclid_km,cwd_between_parts,path_max_cost,path_cells_cost10plus,intervening_nodes,crosses_cost_1000,proposed,reason
0,IPCA · Dene Kʼéh Kusān,ipca,IPCA,3,37090.2;2064.3;79.5,44,130.5,85.8;121.4;48.8,1;1;1,0;0;0,,False,link_locked,rule 4: default -- a named area is a managemen...
1,PA · Liard River Corridor Park,pa,,3,481.9;251.9;79.1,5,62.0,6.0;5.2;133.1,1;1;1,0;0;0,,False,link_locked,rule 4: default -- a named area is a managemen...
2,PA · Nahanni National Park Reserve Of Canada,pa,,2,30020.7;43.2,34,177.1,36.5,1,0,,False,link_locked,rule 4: default -- a named area is a managemen...
3,PA · Nááts’Ihch’Oh National Park Reserve Of Ca...,pa,,2,4382.7;515.9,36,55.4,36.8,1,0,,False,link_locked,rule 4: default -- a named area is a managemen...
4,PA · Tombstone Natural Environment Park,pa,,2,1705.3;343.5,2,24.0,2.8,1,0,,False,merge_parts,rule 1: min gap 2 cells < 3 -- rasterization s...


## ⛔ H7 — HARD STOP. Human review required before anything else runs.

1. Open `audit/audit_objects/multipart_review.csv` (and `node_parts.gpkg` in QGIS if the
   geometry helps).
2. Edit the `proposed` column where the rules got it wrong — one of `merge_parts`,
   `link_locked`, `link_competing`, `no_link` (definitions in the spec, §1 D16 and §2).
3. Fill in the last line: `# reviewed_by: <name>, <date>`.
4. Commit the two CSVs + the gpkg (they are tracked on purpose).

Then run `02_calibrate_baseline.ipynb`. It asserts the signature, copies the signed file into
the run dir, and pins its sha256 in `run_config.json` — later runs must reproduce that hash.
